ARTI308 - Machine Learning

# Credit Card Customer Segmentation Project

In this project, you will use K-Means clustering to segment [credit card customers](https://www.kaggle.com/datasets/arjunbhasin2013/ccdata/data) based on their usage behavior. This is an unsupervised learning problem because the dataset does not contain a target label for customer groups.

You will use the `CC_GENERAL.csv` dataset.

## About the Dataset

The dataset contains customer-level credit card usage behavior. Each row represents one credit card holder, and the columns describe different behavioral variables such as balance, purchases, cash advance, payments, and tenure. The goal is to group similar customers together so that the company can understand different customer segments and design better marketing strategies.

## Import Libraries

**Import the libraries you need for data analysis, visualization, preprocessing, clustering, and evaluation.**

In [144]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

## Get the Data

**Read the `CC_GENERAL.csv` file and save it in a dataframe called `df`.**

In [145]:
 df = pd.read_csv("CC_GENERAL.csv")

**Check the first five rows of the dataset.**

In [146]:
 df.head()

,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,C10001,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,0,2,1000.0,201.802084,139.509787,0.000000,12
1,C10002,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,C10003,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,0,12,7500.0,622.066742,627.284787,0.000000,12
3,C10004,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,1,1,7500.0,0.000000,NaN,0.000000,12
4,C10005,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,0,1,1200.0,678.334763,244.791237,0.000000,12


**Check the shape of the dataset.**

In [147]:
 df.shape

(8950, 18)

**Check basic information about the dataset using `info()`.**

In [148]:
 df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8950 entries, 0 to 8949
Data columns (total 18 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   CUST_ID                           8950 non-null   str    
 1   BALANCE                           8950 non-null   float64
 2   BALANCE_FREQUENCY                 8950 non-null   float64
 3   PURCHASES                         8950 non-null   float64
 4   ONEOFF_PURCHASES                  8950 non-null   float64
 5   INSTALLMENTS_PURCHASES            8950 non-null   float64
 6   CASH_ADVANCE                      8950 non-null   float64
 7   PURCHASES_FREQUENCY               8950 non-null   float64
 8   ONEOFF_PURCHASES_FREQUENCY        8950 non-null   float64
 9   PURCHASES_INSTALLMENTS_FREQUENCY  8950 non-null   float64
 10  CASH_ADVANCE_FREQUENCY            8950 non-null   float64
 11  CASH_ADVANCE_TRX                  8950 non-null   int64  
 12  PURCHASES_TRX    

**Check summary statistics using `describe()`.**

In [ ]:
 df.describe()

## Data Cleaning

The column `CUST_ID` is an identification column. It is not useful for clustering because it does not describe customer behavior.

**Drop the `CUST_ID` column from the dataframe.**

In [ ]:
 df = df.drop("CUST_ID", axis=1)

**Check the missing values in each column.**

In [ ]:
 df.isnull().sum()

Some columns may contain missing values.

Hint: You can handle missing values by either:
- filling them with the mean value
- or dropping the rows that contain missing values

For this project, use mean imputation.

**Fill the missing values with the mean of each column.**

In [ ]:
df.fillna(df.mean(), inplace=True) 

**Check the missing values again to make sure they were handled.**

In [ ]:
df.isnull().sum() 

## Exploratory Data Analysis

Before applying clustering, it is important to understand the data.

**Create histograms for the numerical columns.**

In [ ]:
df.hist(figsize=(20, 20))
plt.show()

**Create a correlation heatmap to understand relationships between the features.**

In [ ]:
plt.figure(figsize=(15, 10))

sns.heatmap(df.corr(), cmap="coolwarm")

plt.title("Correlation Heatmap")
plt.show()

**Create a scatter plot between `BALANCE` and `PURCHASES`.**

In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(x=df["BALANCE"], y=df["PURCHASES"])

plt.title("BALANCE vs PURCHASES")

plt.show()

**Create a scatter plot between `BALANCE` and `CASH_ADVANCE`.**

In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(x=df["BALANCE"], y=df["PURCHASES"])

plt.title("BALANCE vs PURCHASES")

plt.show()

## Feature Scaling

K-Means is a distance-based algorithm. Therefore, feature scaling is very important.

The features in this dataset have very different ranges. For example, `BALANCE`, `PURCHASES`, and `CREDIT_LIMIT` may have large values, while frequency columns are between 0 and 1.

**Use StandardScaler to scale the data. Save the scaled data in a variable called `X_scaled`.**

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(df)

## Choosing K Intuitively

Choosing K is one of the most difficult parts of K-Means.

Since this dataset has many features, it is not easy to visually see the clusters directly.

However, we can still compare different K values using the elbow method and silhouette score.

## Elbow Method

**Create a loop that fits K-Means models for K values from 1 to 10. Save the inertia values in a list called `inertia_values`.**

In [ ]:
inertia_values = []

for k in range(1, 11):

    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)

    kmeans.fit(X_scaled)

    inertia_values.append(kmeans.inertia_)

**Plot the elbow curve.**

In [ ]:
plt.plot(range(1,11), inertia_values, marker='o')

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")

plt.title("Elbow Method")

plt.show()

**Output Interpretation**

Look at the elbow curve and try to identify where the decrease in inertia starts to slow down.

That point can suggest a reasonable value for K.

## Silhouette Score

The silhouette score helps evaluate how well-separated the clusters are.

**Create a loop that calculates the silhouette score for K values from 2 to 10. Save the scores in a list called `silhouette_scores`.**

In [ ]:
silhouette_scores = []

for k in range(2, 11):

    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)

    labels = kmeans.fit_predict(X_scaled)

    score = silhouette_score(X_scaled, labels)

    silhouette_scores.append(score)

**Plot the silhouette scores.**

In [ ]:
plt.figure(figsize=(8,6))

plt.plot(range(2,11), silhouette_scores, marker='o')

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")

plt.title("Silhouette Scores")

plt.show()

**Create a table showing each K value and its silhouette score.**

In [ ]:
silhouette_table = pd.DataFrame({
    "K": range(2,11),
    "Silhouette Score": silhouette_scores
})

silhouette_table

**Output Interpretation**

A higher silhouette score usually means better clustering.

However, do not rely only on the highest value. Also consider whether the chosen K makes sense for customer segmentation.

## Create the Final K-Means Model

**Based on the elbow curve and silhouette scores, choose a final K value. Then train a final K-Means model.**

Use `random_state=42` and `n_init=10`.

In [ ]:
final_kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

clusters = final_kmeans.fit_predict(X_scaled)

**Add the final cluster labels to the original dataframe in a new column called `Cluster`.**

In [ ]:
df["Cluster"] = clusters

**Check the first five rows after adding the cluster labels.**

In [ ]:
df.head()

## Cluster Analysis

Now we need to understand what each cluster means.

**Create a summary table using `groupby()` to show the mean values of each feature for each cluster.**

In [ ]:
cluster_summary = df.groupby("Cluster").mean()

cluster_summary

**Check how many customers are in each cluster.**

In [ ]:
df["Cluster"].value_counts()

## Visualizing the Final Clusters

Since the dataset has many features, we will use PCA to reduce the data into two components only for visualization.

This visualization does not replace the original clustering. It only helps us see the clusters in a 2D plot.

**Use PCA with 2 components and plot the clusters.**

In [ ]:
pca = PCA(n_components=2)

pca_components = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    data=pca_components,
    columns=["PCA1", "PCA2"]
)

pca_df["Cluster"] = clusters

plt.figure(figsize=(10,8))

sns.scatterplot(
    x="PCA1",
    y="PCA2",
    hue="Cluster",
    data=pca_df,
    palette="Set2"
)

plt.title("Customer Segments using PCA")

plt.show()

**Output Interpretation**

The PCA plot gives a simplified 2D view of the clusters.

If the clusters are not perfectly separated, that is normal because the original dataset has many features and the plot only shows two compressed dimensions.

## Final Questions

Answer the following questions:

1. Why is this an unsupervised learning problem?

    This is an unsupervised learning problem because the dataset
    does not contain target labels or predefined customer groups.
    The algorithm finds patterns and groups customers automatically.

2. Why did we remove the `CUST_ID` column?

   We removed the CUST_ID column because it is only an identifier
   and does not describe customer behavior.
   It would not help the clustering process.

3. Which columns had missing values?
   The columns with missing values were identified using:
   df.isnull().sum()

4. How did you handle the missing values?
   Missing values were handled using mean imputation.
   Each missing value was replaced with the mean of its column.

5. Why is scaling important before applying K-Means?
   Scaling is important because K-Means uses distance calculations.
   Features with large values could dominate the clustering process
   if the data is not scaled.


6. Which K value did you choose? Explain your answer using the elbow method and silhouette score.
   K = 3 was chosen based on the elbow method and silhouette score.
   The elbow graph showed a sharp drop after K = 3,
   and the silhouette score was reasonably high at that point.

7. Based on the cluster summary table, describe each customer segment in your own words.
   -Cluster 0:
   Customers with high balances and very high cash advance usage. These customers make fewer purchases        and may rely more on cash advances and borrowed money.
   -Cluster 1:
   High-value customers with high purchases, high credit limits, and frequent transactions. These             customers actively use their credit cards for shopping and regular payments.
   -Cluster 2:
   Customers with lower balances, lower spending, and lower credit limits. These customers appear to be       moderate or low-activity credit card users.

8. Which cluster may represent high-value customers?
   Cluster 1 may represent high-value customers because it has the highest PURCHASES, CREDIT_LIMIT,           PAYMENTS, and transaction activity. These customers actively use their credit cards and spend more.

9. Which cluster may represent customers who rely more on cash advance?
   Cluster 0 appears to rely more on cash advances because it has the highest CASH_ADVANCE values and         CASH_ADVANCE_FREQUENCY. This suggests these customers use their credit cards more for borrowing cash       rather than making purchases.

10. How can a company use these clusters for marketing strategy?
    A company can use these customer segments to create more effective marketing strategies.
    For example:
    -High-value customers can receive premium rewards and loyalty programs.
    -Customers who rely on cash advances may receive financial planning or repayment offers.
    -Low-activity customers can receive promotions to encourage more spending and card usage.
    -Customer segmentation helps companies improve customer satisfaction, increase profits, and design         personalized services.